# 入出力：CSV の要約と保存

対応する本文は [io.tex](../chapters/io.tex) である。コードセルを上から順にShift+Enterで実行する。各問の説明にある値や図と、セルの下に表示される結果を比べる。ファイルを作る問では、その直前に保存先を用意する。

## 問1：CSV を読み、JSON に要約する

引用符内にカンマがある地名も `csv.DictReader` なら1つのフィールドとして読める。空欄は空文字列として得られる。入力が未整列でも最初と最後のイベント時刻を正しく求められるよう、UTC の日時へ変換して最小値と最大値を取る。

実データへ適用する場合は `CSV_PATH` を取得済みカタログへ変更する。人工データは別名で作成する。

CSVは列をカンマで区切った表であり、JSONは名前と値を組にして保存できる形式である。ここでは、3件の日時とマグニチュード（地震の規模を表す値）を読み、要約をJSONへ保存する。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/04_io/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "04_io"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/04_io


In [2]:
CSV_PATH = WORK / "usgs_example.csv"
CSV_PATH.write_text('time,latitude,longitude,mag,place\n2024-01-01T00:00:00.000Z,35.0,140.0,3.2,"Offshore, Japan"\n2024-01-01T00:05:00.000Z,36.0,141.0,,Japan\n2024-01-01T00:12:30.000Z,34.5,139.5,2.8,Tokyo\n', encoding="utf-8")

181

この後のセルにある `assert 条件` は、結果を自動で確認する命令である。条件が違っていればエラーで止まり、条件を満たせば次へ進む。何も表示されなくても、エラーが出なければその確認は通っている。画面の数値や図も、説明に示した値と比べる。

### 1．まず1件を表示する

`csv.DictReader()` は1行分を辞書にし、`list()` は全件をリストへ集める。最初の辞書の `mag` は文字列の `"3.2"`、`place` は `"Offshore, Japan"` となる。次のセルで入力が3件あることも確かめる。

In [3]:
import csv
import json
import math
from datetime import datetime

with CSV_PATH.open(encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))
print("件数:", len(rows))
print(rows[0])

件数: 3
{'time': '2024-01-01T00:00:00.000Z', 'latitude': '35.0', 'longitude': '140.0', 'mag': '3.2', 'place': 'Offshore, Japan'}


### 2．文字列を日時と数値へ変換する

`datetime.fromisoformat()` は、`2024-01-01T00:00:00.000Z` のような文字列を日時へ変換する。末尾の `Z` は協定世界時（UTC）を表す。

マグニチュードの空欄は平均や最小値に使えないので除く。`skipped` は除いた理由ごとの件数を保存する辞書であり、`empty` は空欄、`invalid` は数値へ変換できない文字列、`nonfinite` はNaNや無限大である。`math.isfinite()` は有限な数値かを調べる。実行後、有効値が `[3.2, 2.8]`、空欄が1件となることを確認する。

In [4]:
times = [datetime.fromisoformat(row["time"]) for row in rows]
valid_magnitudes = []
skipped = {"empty": 0, "invalid": 0, "nonfinite": 0}
for row in rows:
    text = row["mag"].strip()
    if not text:
        skipped["empty"] += 1
        continue
    try:
        value = float(text)
    except ValueError:
        skipped["invalid"] += 1
        continue
    if not math.isfinite(value):
        skipped["nonfinite"] += 1
        continue
    valid_magnitudes.append(value)
print("有効値:", valid_magnitudes)
print("除外件数:", skipped)

有効値: [3.2, 2.8]
除外件数: {'empty': 1, 'invalid': 0, 'nonfinite': 0}


### 3．件数・期間・値の範囲を保存する

日時の最小値と最大値を取るため、入力の並び順が変わっても期間は同じになる。`isoformat()` は日時を再び文字列にする。UTCは末尾に `+00:00` と表示される。

`n_events` は全3件、`mag_min` は2.8、`mag_max` は3.2となる。`json.dump()` で保存し、`json.load()` で読み戻して、保存前の辞書と一致することを確かめる。

In [5]:
summary = {
    "n_events": len(rows),
    "first_time": min(times).isoformat() if times else None,
    "last_time": max(times).isoformat() if times else None,
    "mag_min": min(valid_magnitudes) if valid_magnitudes else None,
    "mag_max": max(valid_magnitudes) if valid_magnitudes else None,
    "skipped_magnitudes": skipped,
}
results = WORK / "results"
results.mkdir(parents=True, exist_ok=True)
with (results / "usgs_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, allow_nan=False)
with (results / "usgs_summary.json").open(encoding="utf-8") as f:
    restored = json.load(f)
assert restored == summary
assert (summary["n_events"], summary["mag_min"], summary["mag_max"]) == (3, 2.8, 3.2)
assert summary["first_time"] == "2024-01-01T00:00:00+00:00"
assert summary["last_time"] == "2024-01-01T00:12:30+00:00"
assert rows[0]["place"] == "Offshore, Japan"
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "n_events": 3,
  "first_time": "2024-01-01T00:00:00+00:00",
  "last_time": "2024-01-01T00:12:30+00:00",
  "mag_min": 2.8,
  "mag_max": 3.2,
  "skipped_magnitudes": {
    "empty": 1,
    "invalid": 0,
    "nonfinite": 0
  }
}


`parents=True` は親も作り、`exist_ok=True` は既存のフォルダーを許す。`indent=2` は字下げを2文字にする。`None` は JSON の `null` になる。`allow_nan=False` によって JSON の標準外の数値を保存しようとすると失敗する。

## 問2：中間データを Pickle へ保存する

ここでは元の CSV の文字列を保存する。型を変換する場合はその規則も記録する。`wb` と `rb` の `b` はバイナリモードを表す。Pickle の復元は任意のコードを実行し得るため、自分が保存したファイルだけを読み戻す。

In [6]:
import pickle

preview = [{key: row[key] for key in ["time", "latitude", "longitude", "mag"]} for row in rows[:200]]
path = results / "usgs_preview.pkl"
with path.open("wb") as f:
    pickle.dump(preview, f)
with path.open("rb") as f:
    restored_preview = pickle.load(f)
assert restored_preview == preview and len(restored_preview) == 3
print(len(restored_preview), restored_preview[0])

3 {'time': '2024-01-01T00:00:00.000Z', 'latitude': '35.0', 'longitude': '140.0', 'mag': '3.2'}


## 問3：空欄・変換失敗・非有限値を分けて記録する

`float("NaN")` と `float("inf")` は例外にならない。有限なマグニチュードだけを集計する場合は `math.isfinite` が必要である。今回の入力では空欄が1件、それ以外は0件となる。

NaNは計算結果を通常の数値として表せない値、infは無限大を表す。どちらも `float()` 自体は成功するので、変換時の例外だけでは除けない。`math.isfinite()` を併用する理由はここにある。

次の小さな例では、3.2は有限な値、abcは変換失敗、NaNとinfは変換できても有限ではないことが分かる。続くセルで、元のCSVから数えた除外件数を保存する。

In [7]:
for text in ["3.2", "abc", "NaN", "inf"]:
    try:
        value = float(text)
    except ValueError:
        print(text, ": 数値へ変換できません。")
    else:
        print(text, ": 有限な値か =", math.isfinite(value))

3.2 : 有限な値か = True
abc : 数値へ変換できません。
NaN : 有限な値か = False
inf : 有限な値か = False


In [8]:
assert skipped == {"empty": 1, "invalid": 0, "nonfinite": 0}
(results / "skipped_rows.txt").write_text(
    "\n".join(f"{key}: {value}" for key, value in skipped.items()) + "\n", encoding="utf-8")
print((results / "skipped_rows.txt").read_text(encoding="utf-8"))

empty: 1
invalid: 0
nonfinite: 0

